<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb openai pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata
from openai import OpenAI
from openai.types.responses import Response
from IPython.display import Audio, display
from pathlib import Path
from pydantic import BaseModel, Field
from pypdf import PdfReader
from typing import List, Literal

In [3]:
import re, json

In [4]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

chroma_client = PersistentClient("/content/chromadb")
#chroma_client.delete_collection(name="final_project")
chroma_collection = chroma_client.get_or_create_collection(
    name="final_project",
    embedding_function=OpenAIEmbeddingFunction(api_key=openai_api_key, model_name="text-embedding-3-large")
)

# ако искаме всеки път да получаваме празна DB напр. при експериментиране с много .pdf файлове и не искаме те да си пречат
#  => delete_collection + create_collection
# chroma_collection.delete(ids=sentences_ids)

# Изчитане на текста от .pdf

In [5]:
# функция, на която подаваме път към .pdf документа, изчита съдържанието на файла и ни го връща като string

def extract_text(path_to_pdf: Path) -> str:
  with PdfReader(path_to_pdf) as pdf_reader:
    return "\n".join(page.extract_text() for page in pdf_reader.pages)

# Разделяне на целия .pdf на парчета - по брой символи, по изречения

In [6]:
# функция, на която подаваме текст, брой символи на парчето и брой символи на застъпване - брой символи в края и в началото на парчето,
# които да се припокриват със следващото парче, тъй като парчетата са по брой символи, то тогава може дума да бъде разделена между 2 парчета
# и за да не изпускаме смисъла на тази дума вземаме по 200 букви от началото и края на парчето

def chunk_by_length(text: str, length: int = 1000, overlap: int = 200) -> List[str]:
  chunks = []
  index = 0

  while index < len(text):
    end = index + length
    current_chunk = text[index:end]
    chunks.append(current_chunk)
    index = end - overlap

  return chunks


In [7]:
# функция, която разделя текста на изречения -> всяко парче, ще е отделно изречение
# има ограничение при над 1000 изречения =>
# BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'input': array length must be 2048 or less.", 'type': 'invalid_request_error', 'param': None, 'code': None}} in add.

def chunk_by_sentences(text: str) -> List[str]:
  sentences = []

  sentences = re.split(r'(?<=[.!?])\s+', text)

  return sentences


# Попълване на векторна база данни

In [8]:
# функция, която пълни векторната база chromadb
# Съвет: векторната база да се пълни на batch-ове, защото при над 1000 записа, chromadb ще се опита да изгради вектори за всички записи едновременно
# и може да достигне "лимити", които да попречат завършване на записване в DB

def add_to_chromadb(ids: List[str], chunks: List[str]) -> None:
  batch_index = 0
  batch_lenght = 100

  while batch_index < len(chunks):
    end = batch_index + batch_lenght

    chroma_collection.add(
      ids = ids[batch_index:end],
      documents = chunks[batch_index: end]
    )
    batch_index = end


# Обработка на ToolCalls

In [9]:
class QueryArgs(BaseModel):
    query: str = Field(description="Въпрос на естествен език.")


In [10]:
TOOLS = [
    {
        "type": "function",
        "name": "query",
        "description": "Използвайте този инструмент, за да получавате отговори на въпроси въз основа на съдържанието на базата от знания.",
        "parameters": {
            "type": "object",
            "properties": QueryArgs.model_json_schema()['properties'],
            "required": list(QueryArgs.model_fields.keys()),
            "additionalProperties": False
        },
        "strict": True
    }
]


In [11]:
def query_handler(raw_args: str) -> str:
    query_args = QueryArgs.model_validate(json.loads(raw_args))
    query_result = chroma_collection.query(query_texts=[query_args.query], n_results=3)

    return "\n-----\n".join(doc for doc in query_result['documents'][0])


HANDLERS = {
    "query": query_handler
}

In [12]:
# функция, която връща всички ToolCalls

def extract_tool_calls(response: Response):
    return [output_item for output_item in response.output if output_item.type == 'function_call']

In [13]:
def retrieve_information(prompt: str, verbose: bool = True) -> str:
  conversation =[
      {"role": "system", "content": "You are a helpful assistant that should ask questions about the underlying knowledgebase. Use the \"query\" tool aggresively with various arguments to achive best results."},
      {"role": "user", "content": prompt}
  ]

  ai_args = { "model": "gpt-6-luna", "tools": TOOLS, "reasoning": { "effort": "high"} }
  response = openai_client.responses.create(**ai_args, input=conversation)
  tool_calls = extract_tool_calls(response)

  while tool_calls:
    conversation.extend(response.output)

    if verbose:
      print(f"Found {len(tool_calls)} tools to execute.")

    for tc in tool_calls:
      handler = HANDLERS[tc.name]
      result = handler(tc.arguments)

      print(tc.arguments) # отпечатва какво query извиква toll_call-a

      conversation.append({"type": "function_call_output", "call_id": tc.call_id, "output": result})

    response = openai_client.responses.create(**ai_args, input=conversation)
    tool_calls = extract_tool_calls(response)

  return response.output_text

In [14]:
class IntermediateResponse(BaseModel):
  question: str
  format: Literal['text','image','audio'] # формата на отговора може да е в един от списъка в Literal

In [15]:
def ask_ai(prompt: str, verbose: bool = False) -> None:
  intermediate_response = openai_client.responses.parse(
        model="gpt-6-luna",
        input=[
            { "role": "system", "content": "You are an intermediate AI agent that should determine the real question of a user against some book and the anticipated response format. The default response format is \"text\". If the user expects an \"image\" or \"audio\" to be generated, the question must predispose a creative and determined answer. The response format and the question are independent - the question should be irrelevant of details that expose what the response format is." },
            { "role": "user", "content": prompt }
        ],
        text_format=IntermediateResponse,
        reasoning={ "effort": "high" }
    )

  if verbose:
      print(intermediate_response.output_parsed)

  answer = retrieve_information(intermediate_response.output_parsed.question, verbose)

  if intermediate_response.output_parsed.format == 'text':
      print(answer)
  elif intermediate_response.output_parsed.format == 'image':
      print(answer)
  elif intermediate_response.output_parsed.format == 'audio':
      audio_content = openai_client.audio.speech.create(
          model="tts-1",
          voice="alloy",
          input=answer
      )
      audio_content.stream_to_file("answer.mp3")
      display(Audio("answer.mp3", autoplay=True))
  else:
      raise RuntimeError("Received incorrect response format.")
  #structred output
  #retrieve_information

In [17]:
pdf_content = extract_text("/content/Ivan_Vazov_-_Nemili-nedragi.pdf")

In [18]:
chunks = chunk_by_length(pdf_content)
len(chunks)

198

In [19]:
sentences = chunk_by_sentences(pdf_content)
sentences_ids = [f"{i + 1}" for i in range(len(sentences))]   # генерираме ids, за да създадем уникални записи в DB с поредния номер на sentence

In [20]:
chunks_ids = [f"{i + 1}" for i in range(len(chunks))]   # генерираме ids, за да създадем уникални записи в DB с поредния номер на chunk
documents = [chunk for chunk in chunks]

In [21]:
add_to_chromadb(ids=chunks_ids, chunks=documents)

In [ ]:
ask_ai("Намери изречения, в които се споменава за смърт.", verbose=True)

In [22]:
ask_ai("Намери изречения, в които се споменава за смърт. След това ми ги прочети.", verbose=True)

question='Намери в книгата всички изречения, в които се споменава смъртта.' format='audio'
Found 1 tools to execute.
{"query":"Намери всички изречения в книгата, в които се споменава смъртта, умиране, починал/починала, загинал/загинала, убит/убийство или други форми на думата „смърт“. Върни точните изречения и посочи източник/страница, ако е налична."}
Found 4 tools to execute.
{"query":"В книгата „Немили-недраги“ извлечи всички пълни изречения, съдържащи думата „смърт“ или нейни форми („смъртта“, „смъртна“, „смърти“, „смъртен“). Точни цитати с номерата на страниците."}
{"query":"Намери в цялата книга „Немили-недраги“ изречения за умиране и умиращ човек: умира, умирал, умиращ, умря, умрял, умре, умират. Цитирай точно."}
{"query":"Намери изреченията в книгата, в които се казва, че някой е загинал, убит, погинал или починал, включително смърт в бой. Дай пълните изречения с контекст и страници."}
{"query":"В книгата намери евфемизми и описания на предстояща смърт, последни мигове, гроб, п

/tmp/ipykernel_2219/1079445062.py:27: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  audio_content.stream_to_file("answer.mp3")
